# Case Study 2 — Corn-Ethanol Biofuel Pathway

Companion: **study.md**. Parameterized bioprocess LCA vs a fossil reference on
an energy-equivalent (per-MJ) basis, with DDGS co-product substitution and
Monte Carlo. Foreground parameters come from a CSV — the same interface a
Biosteam simulation would populate.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import bw2data as bd
import bw2calc as bc
import bw2io as bi

13:19:55-0400

 [

warning  

] 

Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.

In [2]:
PROJECT = "cs2-biofuel"
if PROJECT not in bd.projects:
    if "bw25-tutorials" in bd.projects:
        bd.projects.set_current("bw25-tutorials")
        bd.projects.copy_project(PROJECT, switch=True)
    else:
        bi.remote.install_project("ecoinvent-3.10-biosphere", PROJECT)
        bd.projects.set_current(PROJECT)
else:
    bd.projects.set_current(PROJECT)
BIOSPHERE = next(d for d in bd.databases if "biosphere" in d.lower())
bio = bd.Database(BIOSPHERE)
co2 = next(f for f in bio if f["name"] == "Carbon dioxide, fossil" and f["categories"] == ("air",))
print("project:", bd.projects.current)

project:

cs2-biofuel

## Load the bioprocess parameters (the Biosteam-style interface)

In [3]:
params = pd.read_csv(Path.cwd() / "data" / "ethanol_parameters.csv")
print(params.to_string(index=False))
P = dict(zip(params["parameter"], params["value"]))
SCALE = dict(zip(params["parameter"], params["scale"]))

      parameter  value                 unit distribution  scale                                      notes
     corn_yield  0.360 kg ethanol / kg corn    lognormal   0.10          fermentation + distillation yield
        corn_ci  0.300     kg CO2 / kg corn    lognormal   0.15 corn cultivation (fertilizer N2O + diesel)
   process_heat  2.800      MJ / kg ethanol    lognormal   0.12                  distillation steam demand
        heat_ci  0.075          kg CO2 / MJ    lognormal   0.10                         natural gas boiler
       elec_use  0.350     kWh / kg ethanol    lognormal   0.12                            milling + pumps
        grid_ci  0.420         kg CO2 / kWh    lognormal   0.10                           grid electricity
    ddgs_credit  0.450 kg DDGS / kg ethanol   triangular   0.00  co-product (animal feed) for substitution
ddgs_ci_avoided  0.550     kg CO2 / kg DDGS    lognormal   0.20                  avoided soybean-meal feed
   gasoline_ref  3.200     kg CO2 / k

## Build a per-kg-ethanol foreground with the DDGS credit as negative CO2

We fold each upstream burden into a single CO2-equivalent emission per kg
ethanol (a lumped foreground), then convert to per-MJ. The DDGS substitution
credit enters as a NEGATIVE biosphere amount (avoided burden).

In [4]:
def ethanol_co2_per_kg(p, include_ddgs=True):
    corn = (1.0 / p["corn_yield"]) * p["corn_ci"]
    heat = p["process_heat"] * p["heat_ci"]
    elec = p["elec_use"] * p["grid_ci"]
    gross = corn + heat + elec
    credit = p["ddgs_credit"] * p["ddgs_ci_avoided"] if include_ddgs else 0.0
    return gross - credit

def build_ethanol(p, include_ddgs=True):
    DB = "cs2_fg"
    if DB in bd.databases:
        del bd.databases[DB]
    net = ethanol_co2_per_kg(p, include_ddgs)
    bd.Database(DB).write({
        (DB, "ethanol"): {"name": "corn ethanol (per kg)", "unit": "kilogram",
                          "exchanges": [
            {"input": (DB, "ethanol"), "amount": 1.0, "type": "production"},
            {"input": co2.key, "amount": net, "type": "biosphere", "negative": net < 0}]},
    })
    return bd.get_node(database=DB, code="ethanol")

gwp = next(m for m in bd.methods
           if "IPCC 2013" in str(m) and "GWP100" in str(m).replace(" ", "")
           and "no LT" not in str(m) and "SLCF" not in str(m))

# Deterministic cross-check: build in Brightway once and confirm the analytical
# score matches (CO2 CF = 1, so score == net CO2 per kg). Thereafter we use the
# analytical form so the 1000-iteration Monte Carlo does not rebuild the database
# each draw (which would be ~1000 DB writes — minutes instead of milliseconds).
_act = build_ethanol(P, include_ddgs=True)
_l = bc.LCA({_act: 1}, method=gwp); _l.lci(); _l.lcia()
_analytic = ethanol_co2_per_kg(P, include_ddgs=True)
assert abs(_l.score - _analytic) / abs(_analytic) < 1e-4, "analytic vs Brightway mismatch"
print(f"Brightway score {_l.score:.4f} == analytic {_analytic:.4f} kg CO2/kg  ✅")

def per_mj(p, include_ddgs=True):
    # analytical (validated above): net CO2 per kg / energy density
    return ethanol_co2_per_kg(p, include_ddgs) / p["ethanol_energy"]

13:19:56-0400

 [

warning  

] 

Not able to determine geocollections for all datasets. This database is not ready for regionalization.

...

 [

info     

] 

Vacuuming database            

Brightway score 0.9428 == analytic 0.9428 kg CO2/kg  ✅

## LCIA — ethanol (with/without credit) vs gasoline, per MJ

In [5]:
eth_no_credit = per_mj(P, include_ddgs=False)
eth_credit = per_mj(P, include_ddgs=True)
gasoline = P["gasoline_ref"] / P["gasoline_energy"]

comp = pd.DataFrame({
    "fuel": ["ethanol (no co-product credit)", "ethanol (DDGS substitution)", "fossil gasoline"],
    "gCO2e_per_MJ": [eth_no_credit*1000, eth_credit*1000, gasoline*1000],
})
print(comp.round(2).to_string(index=False))
savings = (gasoline - eth_credit) / gasoline
print(f"\nGHG savings vs gasoline (with DDGS credit): {savings:.1%}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(comp["fuel"], comp["gCO2e_per_MJ"],
        color=["#DD8452", "#55A868", "#C44E52"])
ax.set_xlabel("g CO2-eq / MJ fuel"); ax.set_title("Corn ethanol vs fossil gasoline")
plt.tight_layout(); plt.savefig("cs2_comparison.png", dpi=130, bbox_inches="tight")
print("saved cs2_comparison.png"); plt.show()

                          fuel  gCO2e_per_MJ
ethanol (no co-product credit)         44.42
   ethanol (DDGS substitution)         35.18
               fossil gasoline         73.73


GHG savings vs gasoline (with DDGS credit): 52.3%

saved cs2_comparison.png

C:\Users\derne\AppData\Local\Temp\ipykernel_40744\872997019.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  print("saved cs2_comparison.png"); plt.show()


## Contribution breakdown (per kg ethanol, before credit)

In [6]:
corn = (1.0 / P["corn_yield"]) * P["corn_ci"]
heat = P["process_heat"] * P["heat_ci"]
elec = P["elec_use"] * P["grid_ci"]
credit = P["ddgs_credit"] * P["ddgs_ci_avoided"]
contrib = pd.Series({"corn cultivation": corn, "process heat": heat,
                     "electricity": elec, "DDGS credit": -credit})
print(contrib.round(4))
fig, ax = plt.subplots(figsize=(7, 3.5))
colors = ["#4C72B0" if v >= 0 else "#55A868" for v in contrib.values]
ax.bar(contrib.index, contrib.values, color=colors)
ax.axhline(0, color="k", lw=1)
ax.set_ylabel("kg CO2-eq / kg ethanol"); ax.set_title("Contribution (negative = avoided burden)")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.savefig("cs2_contribution.png", dpi=130, bbox_inches="tight")
print("saved cs2_contribution.png"); plt.show()

corn cultivation    0.8333
process heat        0.2100
electricity         0.1470
DDGS credit        -0.2475
dtype: float64

saved cs2_contribution.png

C:\Users\derne\AppData\Local\Temp\ipykernel_40744\394491920.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  print("saved cs2_contribution.png"); plt.show()


## Monte Carlo — propagate CSV distributions to per-MJ savings

In [7]:
rng = np.random.default_rng(7)
def sample(p):
    q = dict(p)
    for k in q:
        s = SCALE.get(k, 0.0)
        if s and s > 0:
            q[k] = p[k] * np.exp(rng.normal(0, s))   # lognormal multiplicative
    return q

N = 1000
eth_draws = np.array([per_mj(sample(P), include_ddgs=True) for _ in range(N)])
gas_draws = np.array([(sample(P)["gasoline_ref"]) / P["gasoline_energy"] for _ in range(N)])
sav_draws = (gas_draws - eth_draws) / gas_draws
p_better = float(np.mean(eth_draws < gas_draws))
print(f"ethanol per-MJ: median={np.median(eth_draws)*1000:.1f} gCO2e/MJ, "
      f"90% CI=[{np.percentile(eth_draws,5)*1000:.1f}, {np.percentile(eth_draws,95)*1000:.1f}]")
print(f"P(ethanol < gasoline) = {p_better:.1%}")
print(f"savings: median={np.median(sav_draws):.1%}, "
      f"90% CI=[{np.percentile(sav_draws,5):.1%}, {np.percentile(sav_draws,95):.1%}]")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(sav_draws*100, bins=40, color="#8172B3")
ax.axvline(np.median(sav_draws)*100, color="k", lw=1)
ax.set_xlabel("GHG savings vs gasoline (%)"); ax.set_ylabel("MC frequency")
ax.set_title(f"Ethanol GHG savings distribution (P better = {p_better:.0%})")
plt.tight_layout(); plt.savefig("cs2_savings_mc.png", dpi=130, bbox_inches="tight")
print("saved cs2_savings_mc.png"); plt.show()

ethanol per-MJ: median=35.1 gCO2e/MJ, 90% CI=[26.1, 46.0]

P(ethanol < gasoline) = 100.0%

savings: median=52.7%, 90% CI=[35.4%, 66.2%]

saved cs2_savings_mc.png

C:\Users\derne\AppData\Local\Temp\ipykernel_40744\849132429.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  print("saved cs2_savings_mc.png"); plt.show()


## Tornado sensitivity (one-at-a-time +-20%)

In [8]:
base = per_mj(P, include_ddgs=True)
tor = []
for k in ["corn_yield", "corn_ci", "process_heat", "ddgs_ci_avoided", "grid_ci"]:
    lo = per_mj({**P, k: P[k]*0.8}, include_ddgs=True)
    hi = per_mj({**P, k: P[k]*1.2}, include_ddgs=True)
    tor.append({"param": k, "low": (lo-base)*1000, "high": (hi-base)*1000,
                "swing": abs(hi-lo)*1000})
tdf = pd.DataFrame(tor).sort_values("swing")
print(tdf.round(2).to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 3))
y = np.arange(len(tdf))
ax.barh(y, tdf["high"], color="#C44E52", label="+20%")
ax.barh(y, tdf["low"], color="#4C72B0", label="-20%")
ax.set_yticks(y); ax.set_yticklabels(tdf["param"]); ax.axvline(0, color="k", lw=1)
ax.set_xlabel("Δ g CO2e/MJ"); ax.set_title("Sensitivity"); ax.legend()
plt.tight_layout(); plt.savefig("cs2_tornado.png", dpi=130, bbox_inches="tight")
print("saved cs2_tornado.png"); plt.show()

          param   low  high  swing
        grid_ci -1.10  1.10   2.19
   process_heat -1.57  1.57   3.13
ddgs_ci_avoided  1.85 -1.85   3.69
        corn_ci -6.22  6.22  12.44
     corn_yield  7.77 -5.18  12.96

saved cs2_tornado.png

C:\Users\derne\AppData\Local\Temp\ipykernel_40744\1312617873.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  print("saved cs2_tornado.png"); plt.show()


## Conclusion + export

In [9]:
out = comp.copy()
out.to_csv("cs2_summary.csv", index=False)
print(out.round(2).to_string(index=False))
print(f"\nCorn ethanol shows ~{savings:.0%} GHG savings vs gasoline WITH DDGS credit;")
print("the co-product method is the dominant methodological lever. Land-use change")
print("is excluded and could materially change this — flagged for follow-up.")
print("\nBiosteam++ hook: replace load of ethanol_parameters.csv with simulator outputs.")

                          fuel  gCO2e_per_MJ
ethanol (no co-product credit)         44.42
   ethanol (DDGS substitution)         35.18
               fossil gasoline         73.73


Corn ethanol shows ~52% GHG savings vs gasoline WITH DDGS credit;

the co-product method is the dominant methodological lever. Land-use change

is excluded and could materially change this — flagged for follow-up.


Biosteam++ hook: replace load of ethanol_parameters.csv with simulator outputs.